# Sequence Motif Search Demo

This notebook demonstrates all the sequence motif search capabilities provided by the `peptide_motif_search` package. We'll cover:

1. **File Conversion** - Converting PDB, CIF, and FASTA files to sequence CSVs
2. **UniProt API Integration** - Retrieving sequences directly from UniProt
3. **Motif Nomenclature** - All available symbols for pattern matching
4. **Motif Searching** - Using the Aho-Corasick algorithm for efficient pattern matching
5. **Analyzing Results** - Interpreting the output JSON and CSV files

## Setup

First, let's import the required modules and set up our paths.

In [ ]:
import sys
import os
import pandas as pd
import json

# Add the sequence_motif directory to the path
sys.path.insert(0, os.path.abspath('../sequence_motif'))

# Import the core modules
from file_converter import process_protein_files
from motif_searcher import run_motif_search, expand_motif, AMINO_ACID_NOMENCLATURE
from uniprot_api import search_uniprot, to_csv

# Define working directories
PROTEIN_FILES_DIR = '../protein_files'
MOTIF_LIBRARIES_DIR = '../sequence_motif/motif_libraries'
OUTPUT_DIR = '../outputs'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Setup complete!")

---
## 1. File Conversion

The `file_converter` module can convert protein structure files (`.pdb`, `.cif`) and sequence files (`.fasta`) into a standardized CSV format.

### Supported Input Formats:
- **PDB files** - Protein Data Bank format
- **CIF files** - Crystallographic Information File format
- **FASTA files** - Sequence files (automatically translates DNA to protein if needed)

In [ ]:
# Let's see what protein files are available
print("Available protein files:")
for f in sorted(os.listdir(PROTEIN_FILES_DIR)):
    if f.endswith(('.pdb', '.cif', '.fasta', '.fa')):
        print(f"  - {f}")

In [ ]:
# Convert protein files to a sequences CSV
output_csv = os.path.join(OUTPUT_DIR, 'converted_sequences.csv')
process_protein_files(PROTEIN_FILES_DIR, output_csv)

print(f"\nConverted sequences saved to: {output_csv}")

In [ ]:
# View the converted sequences
sequences_df = pd.read_csv(output_csv)
print(f"Total sequences: {len(sequences_df)}")
print("\nSample of converted sequences:")
sequences_df['sequence_length'] = sequences_df['sequence'].str.len()
sequences_df[['name', 'sequence_length']].head(10)

---
## 2. UniProt API Integration

You can retrieve protein sequences directly from UniProt using various search criteria:
- **query** - General search string
- **organism** - Filter by organism name
- **enzyme_family** - Filter by enzyme family
- **protein_name** - Filter by protein name
- **accession** - Filter by accession number

In [ ]:
# Example: Fetch human kinase substrate proteins from UniProt
query = '(organism_name:"Homo sapiens") AND (protein_name:"kinase")'
print(f"Querying UniProt with: {query}")

# Fetch a small sample (limit=10 for demo)
data = search_uniprot(query, limit=10)

# Save to CSV
uniprot_csv = os.path.join(OUTPUT_DIR, 'uniprot_sequences.csv')
to_csv(data, uniprot_csv)

In [ ]:
# View the UniProt results
if os.path.exists(uniprot_csv):
    uniprot_df = pd.read_csv(uniprot_csv)
    print(f"Retrieved {len(uniprot_df)} sequences from UniProt")
    uniprot_df[['Entry', 'Protein names', 'organism_name']].head()

---
## 3. Motif Nomenclature

The sequence motif search supports a rich nomenclature for defining complex patterns. This is the core feature that makes the tool powerful for biological pattern discovery.

### Symbol Reference Table

| Symbol | Description | Amino Acids |
|:------:|-------------|-------------|
| `x` | Any amino acid | A, C, D, E, F, G, H, I, K, L, M, N, P, Q, R, S, T, V, W, Y |
| `[ABC]` | Custom set | Any of A, B, or C |
| `{ABC}` | Exclusion set | Any EXCEPT A, B, or C |
| `%` | Hydrophobic | A, V, I, L, M, F, Y, W |
| `@` | Aromatic | F, Y, W, H |
| `&` | Polar | R, N, D, Q, E, K, H, S, T, Y |
| `[+]` | Positively charged | K, R, H |
| `[-]` | Negatively charged | D, E |
| `#` | Aliphatic | A, V, L, I |
| `~` | Small | A, C, D, G, N, P, S, T, V |

### Post-Translational Modifications (PTMs)
| Symbol | Modification | Maps to |
|:------:|--------------|--------|
| `[Y:po]` | Phosphotyrosine | Y |
| `[S:gl]` | Glycosylserine | S |
| `[K:ac]` | Acetyllysine | K |

In [ ]:
# View all defined nomenclature symbols
print("Complete Amino Acid Nomenclature Dictionary:")
print("="*60)
for symbol, residues in AMINO_ACID_NOMENCLATURE.items():
    if len(symbol) <= 3:  # Skip PTM notations for cleaner display
        print(f"  {symbol:5s} -> {', '.join(residues)}")

In [ ]:
# Demonstrate motif expansion
# The expand_motif function converts consensus patterns into all possible concrete sequences

example_motifs = [
    "RRx[ST]",          # PKA consensus (basic example with wildcard and custom set)
    "[DE]xxG",          # Custom set with wildcards
    "{P}G",             # Exclusion: any amino acid except P followed by G
    "%xL",              # Hydrophobic followed by any and then L
    "[+]G",             # Positively charged followed by G
    "[-]xx[+]",         # Negatively charged .. positively charged
]

print("Motif Expansion Examples:")
print("="*60)
for motif in example_motifs:
    processed = motif.replace('-', '')  # Remove dashes (used for readability)
    expanded = expand_motif(processed)
    print(f"\nMotif: {motif}")
    print(f"  Expands to {len(expanded)} concrete sequences")
    print(f"  First 5: {expanded[:5]}")

---
## 4. Motif Searching

The motif search uses the **Aho-Corasick algorithm** for efficient multi-pattern matching. This allows searching for thousands of motifs in a single pass through each sequence.

### Available Motif Libraries:
Let's explore the pre-defined motif libraries included with the package.

In [ ]:
# View available motif libraries
print("Available Motif Libraries:")
print("="*60)
for f in sorted(os.listdir(MOTIF_LIBRARIES_DIR)):
    if f.endswith('.csv'):
        filepath = os.path.join(MOTIF_LIBRARIES_DIR, f)
        try:
            df = pd.read_csv(filepath)
            print(f"  {f}: {len(df)} motifs")
        except:
            print(f"  {f}: (could not read)")

In [ ]:
# Examine the protease motif library
protease_motifs = pd.read_csv(os.path.join(MOTIF_LIBRARIES_DIR, 'protease_sample_motifs.csv'))
print("Protease Cleavage Site Motifs:")
protease_motifs

In [ ]:
# Examine the kinase substrate motif library
kinase_motifs = pd.read_csv(os.path.join(MOTIF_LIBRARIES_DIR, 'kinase_substrate_motifs.csv'))
print("Kinase Substrate Motifs:")
kinase_motifs

### Running a Motif Search

Now let's run the motif search on our converted sequences using the kinase substrate motifs.

In [ ]:
# Run motif search
# This will search for all kinase substrate motifs in our converted sequences

motifs_file = os.path.join(MOTIF_LIBRARIES_DIR, 'kinase_substrate_motifs.csv')
sequences_file = os.path.join(OUTPUT_DIR, 'converted_sequences.csv')

# Change to the sequence_motif directory for output
original_dir = os.getcwd()
os.chdir('../sequence_motif')

run_motif_search(
    motifs_file=motifs_file,
    motif_column='motifs',
    motif_name_column='motif_name',
    sequences_file=sequences_file,
    sequence_column='sequence',
    output_file='motif_results.csv',
    name_column='name'
)

os.chdir(original_dir)

---
## 5. Analyzing Results

The motif search produces two types of output:

1. **Aggregate CSV** - Summary of all findings across all sequences
2. **Individual JSON files** - Detailed results for each sequence

In [ ]:
# Find the most recent results directory
outputs_dir = '../sequence_motif/outputs'
result_dirs = [d for d in os.listdir(outputs_dir) if d.endswith('_motif_search_results_jsons')]
if result_dirs:
    latest_dir = sorted(result_dirs)[-1]
    results_path = os.path.join(outputs_dir, latest_dir)
    print(f"Results directory: {latest_dir}")
    
    # List JSON files
    json_files = [f for f in os.listdir(results_path) if f.endswith('.json')]
    print(f"\nGenerated {len(json_files)} individual result files")
else:
    print("No results found. Please run the motif search first.")

In [ ]:
# Examine an individual JSON result file
if result_dirs and json_files:
    sample_json = os.path.join(results_path, json_files[0])
    with open(sample_json, 'r') as f:
        result = json.load(f)
    
    print(f"Sample Result File: {json_files[0]}")
    print("="*60)
    print(f"Sequence Name: {result['name']}")
    print(f"Sequence Length: {len(result['sequence'])} residues")
    print(f"\nMotifs Found:")
    for motif, data in result['results'].items():
        print(f"  {data['motif_name']} ({motif}):")
        for match in data['matches'][:3]:  # Show first 3 matches
            end_pos, concrete = match
            print(f"    Position {end_pos}: {concrete}")
        if len(data['matches']) > 3:
            print(f"    ... and {len(data['matches']) - 3} more matches")

In [ ]:
# Read the aggregate CSV results
csv_files = [f for f in os.listdir(outputs_dir) if f.endswith('_all_results.csv')]
if csv_files:
    latest_csv = sorted(csv_files)[-1]
    results_df = pd.read_csv(os.path.join(outputs_dir, latest_csv))
    print(f"Aggregate Results: {latest_csv}")
    print(f"Total sequences processed: {len(results_df)}")
    results_df[['name']].head(10)

---
## Custom Motif Search Example

Let's create a custom motif file and run a search.

In [ ]:
# Create a custom motif file
custom_motifs = pd.DataFrame({
    'motif_name': [
        'Ser/Thr_Phosphorylation_Site',
        'RGD_Cell_Adhesion',
        'Nuclear_Localization_Signal',
        'Hydrophobic_Core',
        'Charged_Pair'
    ],
    'motifs': [
        '[ST]P',           # Ser or Thr followed by Pro 
        'RGD',             # Classic cell adhesion motif
        '[+][+]xx[+]',     # Simplified NLS pattern
        '%%%',             # Three hydrophobic residues
        '[-]x[+]'          # Negatively charged, any, positively charged
    ]
})

custom_motifs_file = os.path.join(OUTPUT_DIR, 'custom_motifs.csv')
custom_motifs.to_csv(custom_motifs_file, index=False)
print("Custom motifs created:")
custom_motifs

In [ ]:
# Run search with custom motifs
os.chdir('../sequence_motif')

run_motif_search(
    motifs_file=custom_motifs_file,
    motif_column='motifs',
    motif_name_column='motif_name',
    sequences_file=sequences_file,
    sequence_column='sequence',
    output_file='custom_motif_results.csv',
    name_column='name'
)

os.chdir(original_dir)
print("\nCustom motif search complete!")

---
## Summary

This notebook demonstrated the following sequence motif search features:

1. **File Conversion**
   - Converting PDB, CIF, and FASTA files to CSV
   - Automatic DNA-to-protein translation for FASTA files

2. **UniProt Integration**
   - Retrieving sequences with flexible query parameters
   - Filtering by organism, protein name, accession, etc.

3. **Rich Motif Nomenclature**
   - Standard amino acids (single letters)
   - Wildcards (`x` for any)
   - Custom sets (`[ABC]`) and exclusions (`{ABC}`)
   - Biochemical property groups: `%` (hydrophobic), `@` (aromatic), `&` (polar)
   - Charge-based: `[+]` (positive), `[-]` (negative)
   - Size-based: `#` (aliphatic), `~` (small)
   - PTM support: `[Y:po]`, `[S:gl]`, `[K:ac]`, etc.

4. **Efficient Searching**
   - Aho-Corasick algorithm for multi-pattern matching
   - Handles 150,000+ motifs in seconds

5. **Structured Output**
   - Individual JSON files per sequence
   - Aggregate CSV with all results